In [1]:
import pandas as pd
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
notes_val = pd.read_csv("notes_val_predictions_task4.csv")
ecg_val = pd.read_csv("ecg_val_predictions_task4.csv")
structured_val = pd.read_csv("structured_val_predictions_task4.csv")

display(notes_val.head())
display(ecg_val.head())
display(structured_val.head())

print(len(notes_val))
print(len(ecg_val))
print(len(structured_val))

,sample_id,y_true,pred_prob_notes
0,0,0,0.040617
1,1,0,0.739516
2,2,0,0.170715
3,3,1,0.999067
4,4,0,0.017127


,sample_id,y_true,pred_prob_ecg
0,0,0,0.134709
1,1,0,0.284442
2,2,0,0.274959
3,3,1,0.061741
4,4,0,0.032299


,sample_id,y_true,pred_prob_structured
0,0,0,0.208576
1,1,0,0.182002
2,2,0,0.348863
3,3,1,0.866424
4,4,0,0.059331


18986
18986
18986


In [3]:
val_df = notes_val.merge(
    ecg_val[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_val[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)

val_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,0,0.040617,0.134709,0.208576
1,1,0,0.739516,0.284442,0.182002
2,2,0,0.170715,0.274959,0.348863
3,3,1,0.999067,0.061741,0.866424
4,4,0,0.017127,0.032299,0.059331
...,...,...,...,...,...
18981,18981,0,0.466876,0.052224,0.032643
18982,18982,0,0.517502,0.077467,0.071249
18983,18983,0,0.001744,0.031169,0.062595
18984,18984,0,0.013571,0.032232,0.003004


In [4]:
notes_test = pd.read_csv("notes_test_predictions_task4.csv")
ecg_test = pd.read_csv("ecg_test_predictions_task4.csv")
structured_test = pd.read_csv("structured_test_predictions_task4.csv")

display(notes_test.head())
display(ecg_test.head())
display(structured_test.head())

print(len(notes_test))
print(len(ecg_test))
print(len(structured_test))

,sample_id,y_true,pred_prob_notes
0,0,0,0.121842
1,1,0,0.005888
2,2,0,0.005323
3,3,0,0.045820
4,4,0,0.002736


,sample_id,y_true,pred_prob_ecg
0,0,0,0.035970
1,1,0,0.047330
2,2,0,0.037473
3,3,0,0.122730
4,4,0,0.032845


,sample_id,y_true,pred_prob_structured
0,0,0,0.107265
1,1,0,0.004607
2,2,0,0.004326
3,3,0,0.003726
4,4,0,0.005754


18987
18987
18987


In [5]:
test_df = notes_test.merge(
    ecg_test[["sample_id", "pred_prob_ecg"]],
    on="sample_id",
    how="inner"
).merge(
    structured_test[["sample_id", "pred_prob_structured"]],
    on="sample_id",
    how="inner"
)
test_df

,sample_id,y_true,pred_prob_notes,pred_prob_ecg,pred_prob_structured
0,0,0,0.121842,0.035970,0.107265
1,1,0,0.005888,0.047330,0.004607
2,2,0,0.005323,0.037473,0.004326
3,3,0,0.045820,0.122730,0.003726
4,4,0,0.002736,0.032845,0.005754
...,...,...,...,...,...
18982,18982,1,0.003629,0.092522,0.057156
18983,18983,0,0.039937,0.040239,0.084957
18984,18984,0,0.961425,0.122049,0.122117
18985,18985,0,0.201931,0.167555,0.004373


In [6]:
# Feature engineering
def add_meta_features(df):
    eps = 1e-6
    
    # base probs
    s = df["pred_prob_structured"].astype(float)
    n = df["pred_prob_notes"].astype(float)
    e = df["pred_prob_ecg"].astype(float)
    
    # interaction terms
    df["notes_x_ecg"] = n * e
    df["notes_x_struct"] = n * s
    df["ecg_x_struct"] = e * s
    df["notes_x_ecg_x_struct"] = n * e * s
    
    # pairwise differences
    df["struct_minus_notes"] = s - n
    df["struct_minus_ecg"] = s - e
    df["notes_minus_ecg"] = n - e
    
    # summary stats across modalities
    df["prob_mean"] = (s + n + e) / 3.0
    df["prob_max"] = np.maximum(np.maximum(s, n), e)
    df["prob_min"] = np.minimum(np.minimum(s, n), e)
    df["prob_range"] = df["prob_max"] - df["prob_min"]
    
    # agreement / dispersion
    df["prob_std"] = np.std(np.vstack([s, n, e]), axis=0)
    
    # logit transform can help logistic stacking
    df["logit_struct"] = np.log((s + eps) / (1 - s + eps))
    df["logit_notes"] = np.log((n + eps) / (1 - n + eps))
    df["logit_ecg"] = np.log((e + eps) / (1 - e + eps))
    
    # rank-like indicators / thresholds
    df["struct_gt_notes"] = (s > n).astype(int)
    df["struct_gt_ecg"] = (s > e).astype(int)
    df["notes_gt_ecg"] = (n > e).astype(int)
    
    # confidence-style features
    df["struct_conf"] = np.abs(s - 0.5)
    df["notes_conf"] = np.abs(n - 0.5)
    df["ecg_conf"] = np.abs(e - 0.5)
    
    return df

val_df = add_meta_features(val_df.copy())
test_df = add_meta_features(test_df.copy())

# Features
stack_features = [
    # raw probs
    "pred_prob_structured",
    "pred_prob_notes",
    "pred_prob_ecg",
    
    # interactions
    "notes_x_ecg",
    "notes_x_struct",
    "ecg_x_struct",
    "notes_x_ecg_x_struct",
    
    # differences
    "struct_minus_notes",
    "struct_minus_ecg",
    "notes_minus_ecg",
    
    # summaries
    "prob_mean",
    "prob_max",
    "prob_min",
    "prob_range",
    "prob_std",
    
    # logits
    "logit_struct",
    "logit_notes",
    "logit_ecg",
    
    # comparisons
    "struct_gt_notes",
    "struct_gt_ecg",
    "notes_gt_ecg",
    
    # confidence
    "struct_conf",
    "notes_conf",
    "ecg_conf",
]

X_val = val_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_val = val_df["y_true"].astype(int)

X_test = test_df[stack_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y_test = test_df["y_true"].astype(int)

# Logistic stacking model
meta_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.2,
        C=0.5,
        max_iter=5000,
        class_weight="balanced",
        random_state=42
    ))
])

meta_model.fit(X_val, y_val)

# Predict + evaluate
test_df["pred_logistic_ensemble"] = meta_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, test_df["pred_logistic_ensemble"])
auprc = average_precision_score(y_test, test_df["pred_logistic_ensemble"])

print("Logistic multimodal test AUROC:", round(auc, 6))
print("Logistic multimodal test AUPRC:", round(auprc, 6))

# Inspect learned weights
lr = meta_model.named_steps["lr"]
coefs = pd.DataFrame({
    "feature": stack_features,
    "coef": lr.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive coefficients:")
print(coefs.head(10).to_string(index=False))

print("\nTop negative coefficients:")
print(coefs.tail(10).to_string(index=False))

Logistic multimodal test AUROC: 0.929657
Logistic multimodal test AUPRC: 0.684276

Top positive coefficients:
             feature     coef
        logit_struct 2.877219
         logit_notes 0.837701
            prob_std 0.404622
           logit_ecg 0.239771
            ecg_conf 0.181092
     struct_gt_notes 0.129092
         struct_conf 0.109535
notes_x_ecg_x_struct 0.076929
      notes_x_struct 0.061488
        notes_gt_ecg 0.009992

Top negative coefficients:
             feature      coef
          notes_conf -0.019251
     pred_prob_notes -0.065240
            prob_min -0.074493
     notes_minus_ecg -0.080068
       struct_gt_ecg -0.105076
           prob_mean -0.136678
          prob_range -0.161305
            prob_max -0.176610
pred_prob_structured -0.227932
    struct_minus_ecg -0.271251
